In [1]:
pip install gurobipy

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install pandas openpyxl

Note: you may need to restart the kernel to use updated packages.


In [3]:
import gurobipy as gp
print(gp.gurobi.version())

(12, 0, 2)


In [4]:
#EXERCISE 1
from gurobipy import Model, GRB, quicksum

#data given:
teachers = ["Γεσμανίδης", "Ινσουλίνα", "Χαρτούλα", "Λαθοπράξης", "Αντιπαράγωγος", "Κιρκοφίδου", "Πλατιάζων", "Μπρατσάκης", "Τρεχαλητούλα"]
subjects = ["Αγγλικά", "Βιολογία", "Ιστορία", "Μαθηματικά", "Φυσική", "Φιλοσοφία", "Φ. Αγωγή"]
days = ["Δευτέρα", "Τρίτη", "Τετάρτη", "Πέμπτη", "Παρασκευή"]
classrooms = ["Τμήμα 1", "Τμήμα 2"]
hours = [0, 1, 2, 3]  # 8-10, 10:15-12:15, 14-16, 16:15-18:15
#συνολικές απαιτούμενες ΄ωρες καθηγητή/τμήμα
demand = { # (teacher, class): # of lessons
    ("Γεσμανίδης", "Τμήμα 1"): 1, ("Γεσμανίδης", "Τμήμα 2"): 1,
    ("Ινσουλίνα", "Τμήμα 1"): 3, ("Ινσουλίνα", "Τμήμα 2"): 3,
    ("Χαρτούλα", "Τμήμα 1"): 2, ("Χαρτούλα", "Τμήμα 2"): 2,
    ("Λαθοπράξης", "Τμήμα 1"): 0, ("Λαθοπράξης", "Τμήμα 2"): 4,
    ("Αντιπαράγωγος", "Τμήμα 1"): 4, ("Αντιπαράγωγος", "Τμήμα 2"): 0,
    ("Κιρκοφίδου", "Τμήμα 1"): 3, ("Κιρκοφίδου", "Τμήμα 2"): 3,
    ("Πλατιάζων", "Τμήμα 1"): 1, ("Πλατιάζων", "Τμήμα 2"): 1,
    ("Μπρατσάκης", "Τμήμα 1"): 1, ("Μπρατσάκης", "Τμήμα 2"): 0,
    ("Τρεχαλητούλα", "Τμήμα 1"): 0, ("Τρεχαλητούλα", "Τμήμα 2"): 1,
}

#μοντέλο:
m = Model("Schedule")

#διάνυσμα μεταβλητ΄ων:
x = m.addVars(teachers, classrooms, days, hours, vtype = GRB.BINARY, name = "x")

#να διδάσκονται όλα τα απαιτούμενα 2ωρα
for p in teachers:
    for t in classrooms:
        if (p, t) in demand:
            m.addConstr(quicksum(x[p, t, d, h] for d in days for h in hours) == demand[(p, t)])
            
#γυμναστική κάθε Πέμπτη 14-16 => ζώνη 2 δηλαδή 3η ώρα
for p in ["Μπρατσάκης", "Τρεχαλητούλα"]:
    for t in classrooms:
        for d in days:
            for h in hours:
                if not (d == "Πέμπτη" and h == 2):  # σωστό
                    m.addConstr(x[p, t, d, h] == 0) #εδώ λέω ότι αν δεν είναι Πέμπτη 14-16 αυτοί οι δυο δεν διδάσκουν αλλά το παρακάμπτω παρακάτω
                    #σημείωση***** μπορώ αυθαίρετα να τους δώσω από ένα τμήμα εκείνη την ώρα

m.addConstr(x["Μπρατσάκης", "Τμήμα 1", "Πέμπτη", 2] == 1)
m.addConstr(x["Τρεχαλητούλα", "Τμήμα 2", "Πέμπτη", 2] == 1) #μπορώ και αντίστροφα τα τμήματα

#Δευτέρα 8-10 δεν γίνονται μαθήματα (ζώνη 0)
for p in teachers:
    for t in classrooms:
        m.addConstr(x[p, t, "Δευτέρα", 0] == 0)

#Λαθοπράξης λείπει Δευτέρα πρωί (ζώνη 0,1) 
#η ζώνη 0 είναι ήδη κενή λόγω του πάνω constraint οπότε:
for t in classrooms:
    m.addConstr(x["Λαθοπράξης", t, "Δευτέρα", 1] == 0)

#Ινσουλίνα δεν εργάζεται Τετάρτη:
for t in  classrooms:
    for h in hours:
        m.addConstr(x["Ινσουλίνα", t, "Τετάρτη", h] == 0)

#κάθε τμήμα έχει μόνο ένα δίωρο ανά μάθημα κάθε μέρα:
for p in teachers:
    for t in classrooms:
        for d in days:
            m.addConstr(quicksum( x[p, t, d, h] for h in hours) <= 1)
#το παραπάνω ορίζει μόνο πως ένας καθηγητής μπορεί να διδάξει max 1 δίωρο/ημέρα σε κάθε τμήμα
#αυτό όμως μας καλύπτει διότι βλέπουμε ότι κάθε μάθημα για ένα συγκεκριμένο τμήμα διδάσκεται από έναν και μόνο έναν καθηγητή
#θα χρειαζόμασταν κι άλλο constraint αν π.χ. ο "Λαθοπράξης¨ δίδασκε 2 δίωρα στο τμήμα 1 και η "Αντιπαράγωγος" τα υπόλοιπα 2

#προφανώς κάθε τμήμα διδάσκεται από max 1 καθηγητή κάθε στιγμή:
for t in classrooms:
    for d in days:
        for h in hours:
            m.addConstr(quicksum(x[p, t, d, h] for p in teachers) <= 1)

#παρομοίως κάθε καθηγητής διδάσκει max 1 τμήμα ανά στιγμή:
for p in teachers:
    for d in days:
        for h in hours:
            m.addConstr(quicksum(x[p, t, d, h] for t in classrooms) <= 1)


Set parameter Username
Set parameter LicenseID to value 2670590
Academic license - for non-commercial use only - expires 2026-05-25


In [5]:
# Αντικειμενική: dummy στόχος (απλώς για feasibility)
m.setObjective(0, GRB.MINIMIZE)

m.optimize()

# Εμφάνιση αποτελέσματος
for p in teachers:
    for t in classrooms:
        for d in days:
            for h in hours:
                if x[p, t, d, h].X > 0.5:
                    print(f"{p} - {t} - {d} - Ώρα {h}")


Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (win64 - Windows 11.0 (26100.2))

CPU model: AMD Ryzen 7 3700U with Radeon Vega Mobile Gfx, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 434 rows, 360 columns and 1546 nonzeros
Model fingerprint: 0x75b07848
Variable types: 0 continuous, 360 integer (360 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [0e+00, 0e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 4e+00]
Found heuristic solution: objective 0.0000000

Explored 0 nodes (0 simplex iterations) in 0.01 seconds (0.00 work units)
Thread count was 1 (of 8 available processors)

Solution count 1: 0 

Optimal solution found (tolerance 1.00e-04)
Best objective 0.000000000000e+00, best bound 0.000000000000e+00, gap 0.0000%
Γεσμανίδης - Τμήμα 1 - Παρασκευή - Ώρα 2
Γεσμανίδης - Τμήμα 2 - Πέμπτη - Ώρα 3
Ινσουλίνα - Τμήμα 1 - Δευτέρα - Ώρα 3
Ινσουλίνα - Τ

In [7]:
#ΠΕΡΝΑΩ ΤΟ ΑΠΟΤΕΛΕΣΜΑ ΑΠΕΥΘΕΙΑΣ ΣΕ EXCEL (αυτό το part το πήρα ψιλοέτοιμο
import pandas as pd

schedule = []

for p in teachers:
    for t in classrooms:
        for d in days:
            for h in hours:
                if x[p, t, d, h].X > 0.5:
                    # mapping ζώνες -> ώρες
                    time_map = {0: "08:00-10:00", 1: "10:15-12:15", 2: "14:00-16:00", 3: "16:15-18:15"}
                    schedule.append({
                        "Καθηγητής": p,
                        "Τμήμα": t,
                        "Ημέρα": d,
                        "Ζώνη": h,
                        "Ώρα": time_map[h]
                    })

df = pd.DataFrame(schedule)

df = df.sort_values(by=["Τμήμα", "Ημέρα", "Ζώνη"])

df.to_excel("schedule1.xlsx", index=False)


✅ Schedule saved to 'school_schedule.xlsx'
